# Semana 40 · ¿Hay más galaxias aparte de la nuestra?

Este cuaderno acompaña a la sesión de proyecto de la semana 40: la regresión lineal por mínimos cuadrados, el coeficiente de correlación y la ley de Leavitt para medir la distancia a una galaxia.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Mínimos cuadrados, a mano

Programamos las fórmulas de la sesión 1 y las comparamos con `np.polyfit`, que hace lo mismo por dentro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ajuste_lineal(x, y):
    """Devuelve la ordenada a, la pendiente b, el coeficiente de correlación r y sus incertidumbres."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    xm, ym = x.mean(), y.mean()
    sxx = ((x - xm) ** 2).sum()
    sxy = ((x - xm) * (y - ym)).sum()
    syy = ((y - ym) ** 2).sum()
    b = sxy / sxx
    a = ym - b * xm
    r = sxy / np.sqrt(sxx * syy)
    residuos = y - (a + b * x)
    s = np.sqrt((residuos ** 2).sum() / (len(x) - 2))
    sigma_b = s / np.sqrt(sxx)
    sigma_a = s * np.sqrt(1 / len(x) + xm**2 / sxx)
    return a, b, r, sigma_a, sigma_b, residuos

x = [1, 2, 3, 4]
y = [2, 3, 5, 6]
a, b, r, sa, sb, res = ajuste_lineal(x, y)
print(f"a mano:     a = {a:.3f} ± {sa:.3f}, b = {b:.3f} ± {sb:.3f}, r = {r:.3f}")
print(f"np.polyfit: {np.polyfit(x, y, 1)[::-1]}")
print(f"residuos: {res}, suman {res.sum():.1e}")

## 2. Correlaciones de todo tipo

Cuatro nubes de puntos con su coeficiente de correlación. El último caso es una advertencia: $r \approx 0$ no significa que no haya relación, solo que no es **lineal**.

In [ ]:
rng = np.random.default_rng(40)
x = rng.uniform(-3, 3, 200)
casos = [("fuerte y positiva", 2 * x + rng.normal(0, 1, 200)),
         ("débil", 0.5 * x + rng.normal(0, 2, 200)),
         ("negativa", -1.5 * x + rng.normal(0, 1.5, 200)),
         ("no lineal", x**2 + rng.normal(0, 0.5, 200))]
fig, axs = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (nombre, y) in zip(axs, casos):
    a, b, r, *_ = ajuste_lineal(x, y)
    ax.scatter(x, y, s=6); xs = np.array([-3, 3]); ax.plot(xs, a + b * xs, "r")
    ax.set_title(f"{nombre}: r = {r:.2f}")
plt.tight_layout(); plt.show()

## 3. Calibrar la ley de Leavitt (simulación)

Generamos 40 cefeidas **simuladas** de la Vía Láctea, con periodos entre 3 y 40 días, cuyas magnitudes absolutas siguen la relación de Benedict y colaboradores (2007), $M_V = -2{,}43\,(\log P - 1) - 4{,}05$, con una dispersión real de 0,2 magnitudes y un pequeño error de medida. Imagina que sus distancias se han medido con Gaia. Luego ajustamos la recta como si no la conociéramos.

In [ ]:
def leavitt(logP):
    return -2.43 * (logP - 1) - 4.05

n = 40
logP = rng.uniform(np.log10(3), np.log10(40), n)
M_obs = leavitt(logP) + rng.normal(0, 0.20, n) + rng.normal(0, 0.05, n)

a, b, r, sa, sb, res = ajuste_lineal(logP, M_obs)
print(f"pendiente = {b:.2f} ± {sb:.2f}   (verdadera: -2,43)")
print(f"ordenada en log P = 1: {a + b:.2f} ± {np.std(res, ddof=2) / np.sqrt(n):.2f}   (verdadera: -4,05)")
print(f"r = {r:.3f};  dispersión de los residuos: {np.std(res, ddof=2):.2f} mag")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
a1.scatter(logP, M_obs, s=15)
lp = np.linspace(0.4, 1.7, 2)
a1.plot(lp, a + b * lp, "r", label="ajuste"); a1.plot(lp, leavitt(lp), "k--", lw=0.8, label="relación verdadera")
a1.invert_yaxis(); a1.set_ylabel("M_V"); a1.legend(); a1.grid(alpha=0.3); a1.set_title("Ley de Leavitt (cefeidas simuladas)")
a2.axhline(0, color="gray"); a2.scatter(logP, res, s=15)
a2.set_xlabel("log₁₀ P (días)"); a2.set_ylabel("residuo (mag)"); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. La distancia a una galaxia

Ahora simulamos 25 cefeidas en una galaxia a 765 kpc (como Andrómeda), con su magnitud aparente media, y estimamos la distancia con la recta calibrada. Repetimos la calibración muchas veces para ver cuánto varía el resultado: es una forma sencilla de propagar los errores, el **método de Monte Carlo**.

In [ ]:
d_real = 765_000                                   # pc
mu_real = 5 * np.log10(d_real / 10)
nG = 25
logP_G = rng.uniform(np.log10(10), np.log10(60), nG)
m_G = leavitt(logP_G) + mu_real + rng.normal(0, 0.20, nG) + rng.normal(0, 0.08, nG)

def distancia_galaxia(a, b):
    mu = np.mean(m_G - (a + b * logP_G))
    return 10 ** (mu / 5 + 1)

print(f"distancia estimada: {distancia_galaxia(a, b) / 1e3:.0f} kpc (real: {d_real / 1e3:.0f} kpc)")

estimaciones = []
for _ in range(2000):
    lp = rng.uniform(np.log10(3), np.log10(40), n)
    Mo = leavitt(lp) + rng.normal(0, 0.20, n) + rng.normal(0, 0.05, n)
    aa, bb, *_ = ajuste_lineal(lp, Mo)
    estimaciones.append(distancia_galaxia(aa, bb))
estimaciones = np.array(estimaciones) / 1e3
plt.figure(figsize=(7, 4))
plt.hist(estimaciones, bins=50, color="tab:green"); plt.axvline(d_real / 1e3, color="r")
plt.xlabel("distancia estimada (kpc)"); plt.title("2000 calibraciones distintas"); plt.show()
print(f"media {estimaciones.mean():.0f} kpc, desviación típica {estimaciones.std():.0f} kpc ({100 * estimaciones.std() / estimaciones.mean():.1f} %)")

El histograma muestra la incertidumbre que viene solo de la calibración, de un 2 % aproximadamente: las 25 cefeidas de la galaxia son siempre las mismas, y su propia dispersión añade otro poco (por eso la media no cae justo en 765 kpc). Además, todas las estimaciones de la galaxia comparten el mismo punto de partida, la calibración: si estuviera mal (como le pasó a Hubble), todas se desplazarían juntas. Es un **error sistemático**, que no baja promediando.

## 5. El peligro de extrapolar

Fíjate en que calibramos con periodos de 3 a 40 días y aplicamos la recta a cefeidas de 10 a 60 días: algunas están fuera del rango calibrado. Si la relación real se curvara un poco fuera de ese rango, la recta daría magnitudes erróneas. Lo simulamos con una relación que se curva levemente para periodos largos.

In [ ]:
def leavitt_curvada(logP):
    return leavitt(logP) - 0.8 * np.clip(logP - 1.6, 0, None) ** 2 * 10

lp = np.linspace(0.4, 2.2, 200)
plt.figure(figsize=(8, 4.5))
plt.plot(lp, leavitt(lp), "k--", label="recta calibrada")
plt.plot(lp, leavitt_curvada(lp), "r", label="una relación real que se curvara")
plt.axvspan(np.log10(3), np.log10(40), color="green", alpha=0.1, label="rango calibrado")
plt.gca().invert_yaxis(); plt.xlabel("log₁₀ P"); plt.ylabel("M_V"); plt.legend(); plt.grid(alpha=0.3); plt.show()
for P in [30, 60, 100]:
    dM = leavitt_curvada(np.log10(P)) - leavitt(np.log10(P))
    print(f"P = {P:3d} días: error en M = {dM:+.2f} mag → error en la distancia del {100 * (10 ** (-0.2 * dM) - 1):+.0f} %")

La curvatura es inventada, pero el aviso es real: una recta ajustada solo es fiable dentro del rango de los datos que la calibran.

## 6. Experimenta

1. Calibra con solo 10 cefeidas. ¿Cuánto crece la incertidumbre de la distancia?
2. Calibra solo con periodos entre 5 y 8 días (poco repartidos). ¿Qué le pasa a la incertidumbre de la pendiente? Compáralo con la fórmula $\sigma_b = s/\sqrt{\sum(x_i - \bar x)^2}$.
3. Añade a la galaxia una extinción de 0,5 magnitudes que no corriges. ¿Qué distancia sale?

In [ ]:
# Tu experimento aquí